# PCA는 선형 뉴런, MLP는 비선형 매니폴드 언폴딩?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gshong-ai/ADA26/blob/claude/mnist-dimensionality-reduction-H2Meg/04_autoencoder_manifold/autoencoder_swiss_roll.ipynb)

## 개요

**핵심 질문:**  
> PCA가 linear activation 하나짜리 뉴런이라면,  
> ReLU를 쓰는 MLP로 Swiss Roll을 2D에 투영하면 Isomap처럼 펼쳐질 수 있을까?

### PCA ↔ 선형 오토인코더의 동치성

```
PCA(2D):           3D ──[W·x]──▶ 2D  (선형, 분산 최대화)

선형 오토인코더:   3D ──[encode: Dense(2, linear)]──▶ 2D
                       ──[decode: Dense(3, linear)]──▶ 3D  (재구성 손실 최소화)
```

이론적으로 **선형 오토인코더의 bottleneck 표현 = PCA의 주성분 공간**입니다 (Baldi & Hornik, 1989).

### 실험 설계

| 모델 | 인코더 구조 | 활성화 | 비고 |
|------|------------|--------|------|
| PCA | — | 선형 (분석해) | 기준선 |
| 선형 AE | 3→2→3 | 모두 linear | PCA와 이론적 동치 |
| MLP-4 | 3→**4**→2→**4**→3 | ReLU (hidden) | 비선형 용량 시작 |
| MLP-8 | 3→**8**→2→**8**→3 | ReLU (hidden) | 중간 용량 |
| MLP-16 | 3→**16**→2→**16**→3 | ReLU (hidden) | 높은 용량 |
| Isomap | — | — (측지 거리) | 비선형 참조 기준 |

**Swiss Roll 3D → 2D 투영 후 색상(매니폴드 위치 t)의 연속성으로 평가합니다.**

## 0. 패키지 설치

In [ ]:
!pip install -q plotly scikit-learn tensorflow

## 1. 라이브러리 임포트

In [ ]:
# ─── 라이브러리 임포트 ───
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

pio.renderers.default = 'colab'

print(f'TensorFlow: {tf.__version__}')
print('라이브러리 로드 완료')

## 2. 하이퍼파라미터 & 설정

In [ ]:
# ─── 하이퍼파라미터 & 전역 설정 ───
RANDOM_SEED   = 42
N_SAMPLES     = 2000      # Swiss Roll 샘플 수
SWISS_NOISE   = 0.1       # Swiss Roll 노이즈 (작게: 매니폴드 구조 선명)

# ─── 오토인코더 학습 설정 ───
EPOCHS        = 300
BATCH_SIZE    = 128
LEARNING_RATE = 1e-3
LATENT_DIM    = 2         # 투영 목표 차원

# ─── 실험할 hidden node 수 ───
HIDDEN_SIZES  = [4, 8, 16]

# ─── Isomap 설정 ───
ISOMAP_N_NEIGHBORS = 10

# ─── 시각화 ───
OUTPUT_HTML   = 'autoencoder_swiss_roll.html'
COLORSCALE    = 'Rainbow'   # 매니폴드 위치 t 표현 색상

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print('설정 완료')

## 3. 데이터 로드 & 전처리

Swiss Roll: 2D 평면이 말린 3D 구조. 색상(t)이 연속적으로 변하면 매니폴드 구조가 잘 보존된 것.

In [ ]:
# ─── Swiss Roll 생성 ───
X_raw, t = make_swiss_roll(n_samples=N_SAMPLES, noise=SWISS_NOISE, random_state=RANDOM_SEED)

# 표준화 (오토인코더 학습 안정성을 위해)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw).astype('float32')

print(f'Swiss Roll shape: {X.shape}')
print(f'매니폴드 파라미터 t 범위: {t.min():.2f} ~ {t.max():.2f}')

# 원본 3D 시각화
fig_3d = go.Figure(go.Scatter3d(
    x=X_raw[:, 0], y=X_raw[:, 1], z=X_raw[:, 2],
    mode='markers',
    marker=dict(size=3, color=t, colorscale=COLORSCALE, opacity=0.85,
                colorbar=dict(title='t (매니폴드 위치)')),
))
fig_3d.update_layout(
    title='원본 Swiss Roll (3D) — 색상이 연속적일수록 구조 보존 잘 된 것',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    width=650, height=500,
)
fig_3d.show()

## 4. 모델 정의 (Keras Functional API)

```
입력(3D)
  │
  ├─ [선형 AE]   Dense(2, linear)  ──────────────────▶ bottleneck(2D)
  │
  └─ [MLP-N]    Dense(N, relu) → Dense(2, linear)  ──▶ bottleneck(2D)
                    ↑
             비선형 표현 학습 공간
```

bottleneck(2D) → 디코더 → 재구성(3D)  
손실: MSE (재구성 오차 최소화)

In [ ]:
# ─── 오토인코더 빌더 함수 ───
def build_autoencoder(input_dim, latent_dim, hidden_size=None):
    """
    hidden_size=None  → 선형 오토인코더 (PCA와 이론적 동치)
    hidden_size=N     → MLP 오토인코더 (ReLU hidden layer)
    """
    # ── 인코더 ──
    inputs = keras.Input(shape=(input_dim,), name='입력')

    if hidden_size is None:
        # 선형 오토인코더: 히든 레이어 없음
        encoded = layers.Dense(latent_dim, activation='linear', name='bottleneck')(inputs)
    else:
        # MLP 오토인코더: ReLU 히든 → 선형 bottleneck
        h_enc = layers.Dense(hidden_size, activation='relu', name=f'인코더_hidden_{hidden_size}')(inputs)
        encoded = layers.Dense(latent_dim, activation='linear', name='bottleneck')(h_enc)

    encoder = keras.Model(inputs, encoded, name='encoder')

    # ── 디코더 ──
    latent_inputs = keras.Input(shape=(latent_dim,), name='잠재공간')

    if hidden_size is None:
        decoded = layers.Dense(input_dim, activation='linear', name='재구성')(latent_inputs)
    else:
        h_dec = layers.Dense(hidden_size, activation='relu', name=f'디코더_hidden_{hidden_size}')(latent_inputs)
        decoded = layers.Dense(input_dim, activation='linear', name='재구성')(h_dec)

    decoder = keras.Model(latent_inputs, decoded, name='decoder')

    # ── 전체 오토인코더 ──
    ae_output = decoder(encoder(inputs))
    autoencoder = keras.Model(inputs, ae_output, name='autoencoder')

    return autoencoder, encoder


# ─── 모델 구조 확인 ───
print('=== 선형 오토인코더 구조 (PCA와 동치) ===')
ae_linear, enc_linear = build_autoencoder(input_dim=3, latent_dim=LATENT_DIM, hidden_size=None)
ae_linear.summary()

print('\n=== MLP-16 오토인코더 구조 ===')
ae_mlp16, enc_mlp16 = build_autoencoder(input_dim=3, latent_dim=LATENT_DIM, hidden_size=16)
ae_mlp16.summary()

## 5. 학습

In [ ]:
# ─── 학습 실행 함수 ───
def train_autoencoder(autoencoder, X_data, label, epochs=EPOCHS):
    """오토인코더 학습 후 학습 곡선과 최종 손실 반환"""
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='mse',
    )
    history = autoencoder.fit(
        X_data, X_data,
        epochs=epochs,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        verbose=0,
        callbacks=[
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                              patience=20, min_lr=1e-5, verbose=0),
        ],
    )
    final_loss = history.history['val_loss'][-1]
    print(f'  [{label}] 최종 검증 손실(MSE): {final_loss:.6f}')
    return history


# ─── 전체 모델 학습 ───
results = {}   # 결과 저장: {label: {'encoder': ..., 'history': ..., 'coords': ...}}

print('=== 오토인코더 학습 시작 ===')
print(f'  Epochs={EPOCHS}, BatchSize={BATCH_SIZE}, LR={LEARNING_RATE}')
print()

# 1. 선형 오토인코더
ae_lin, enc_lin = build_autoencoder(3, LATENT_DIM, hidden_size=None)
hist_lin = train_autoencoder(ae_lin, X, label='선형 AE')
results['선형 AE\n(= PCA)'] = {'encoder': enc_lin, 'history': hist_lin}

# 2. MLP 오토인코더 (hidden_size: 4, 8, 16)
for n in HIDDEN_SIZES:
    ae_n, enc_n = build_autoencoder(3, LATENT_DIM, hidden_size=n)
    hist_n = train_autoencoder(ae_n, X, label=f'MLP-{n}')
    results[f'MLP-{n}\n(hidden={n})'] = {'encoder': enc_n, 'history': hist_n}

print('\n학습 완료')

## 6. 기준선: PCA & Isomap 투영

In [ ]:
# ─── PCA 2D 투영 (이론적 기준선) ───
pca = PCA(n_components=2, random_state=RANDOM_SEED)
coords_pca = pca.fit_transform(X)
print(f'PCA 설명분산: {pca.explained_variance_ratio_.sum():.3f}')

# ─── Isomap 2D 투영 (비선형 목표 기준선) ───
print(f'Isomap 투영 중... (n_neighbors={ISOMAP_N_NEIGHBORS})')
isomap = Isomap(n_components=2, n_neighbors=ISOMAP_N_NEIGHBORS)
coords_isomap = isomap.fit_transform(X)
print(f'Isomap 재구성 오차: {isomap.reconstruction_error():.4f}')

# ─── 인코더로 2D 좌표 추출 ───
for label, d in results.items():
    d['coords'] = d['encoder'].predict(X, verbose=0)

print('\n모든 투영 완료')

## 7. 평가 & 시각화

### 평가 기준
**색상(t)이 무지개처럼 연속적으로 변하면** → 매니폴드 구조가 잘 펼쳐진 것  
색상이 섞여있으면 → 인접한 매니폴드 층이 겹쳐 구조 파괴

In [ ]:
# ─── 2D 투영 비교 시각화 (2×3 그리드) ───

# 표시 순서: PCA → 선형AE → MLP-4 → MLP-8 → MLP-16 → Isomap
plot_items = [
    ('PCA\n(선형, 분석해)', coords_pca),
] + [
    (label, d['coords']) for label, d in results.items()
] + [
    ('Isomap\n(측지 거리 기준)', coords_isomap),
]

n_plots = len(plot_items)   # 6
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[label.replace('\n', ' ') for label, _ in plot_items],
    horizontal_spacing=0.06,
    vertical_spacing=0.12,
)

# 서브플롯 위치 매핑
positions = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)]

for (label, coords), (row, col) in zip(plot_items, positions):
    fig.add_trace(
        go.Scatter(
            x=coords[:, 0],
            y=coords[:, 1],
            mode='markers',
            marker=dict(
                size=4,
                color=t,
                colorscale=COLORSCALE,
                opacity=0.8,
                showscale=(row == 1 and col == 3),   # 오른쪽 상단에만 컬러바
                colorbar=dict(title='t', x=1.02) if (row == 1 and col == 3) else None,
            ),
            showlegend=False,
            hovertemplate='x: %{x:.3f}<br>y: %{y:.3f}<extra></extra>',
        ),
        row=row, col=col,
    )

fig.update_layout(
    title=dict(
        text=(
            'Swiss Roll 2D 투영 비교: PCA → 선형AE → MLP(4·8·16) → Isomap<br>'
            '<sup>색상(t)이 무지개처럼 연속적일수록 매니폴드 구조가 잘 펼쳐진 것</sup>'
        ),
        x=0.5, font=dict(size=16),
    ),
    width=1100, height=780,
    paper_bgcolor='white',
)

fig.show()

fig.write_html(OUTPUT_HTML, include_plotlyjs='cdn')
print(f'✅ {OUTPUT_HTML} 저장 완료')

## 8. 학습 곡선 비교

노드 수가 늘어날수록 재구성 손실이 어떻게 달라지는지 확인합니다.

In [ ]:
# ─── 학습 곡선 비교 ───
loss_colors = {
    '선형 AE\n(= PCA)':     '#95a5a6',
    'MLP-4\n(hidden=4)':  '#3498DB',
    'MLP-8\n(hidden=8)':  '#2ECC71',
    'MLP-16\n(hidden=16)': '#E74C3C',
}

fig_loss = go.Figure()
for label, d in results.items():
    val_loss = d['history'].history['val_loss']
    fig_loss.add_trace(go.Scatter(
        y=val_loss,
        mode='lines',
        name=label.replace('\n', ' '),
        line=dict(color=loss_colors[label], width=2),
    ))

fig_loss.update_layout(
    title='오토인코더 검증 손실(MSE) 학습 곡선 — hidden 노드 수별 비교',
    xaxis_title='Epoch',
    yaxis_title='Validation MSE (재구성 오차)',
    yaxis_type='log',
    width=800, height=450,
    legend=dict(x=0.75, y=0.95),
)
fig_loss.show()

# ─── 최종 손실 요약 ───
print('=== 최종 검증 MSE 요약 ===')
for label, d in results.items():
    final = d['history'].history['val_loss'][-1]
    n_params = sum([np.prod(v.shape) for v in d['encoder'].trainable_weights])
    print(f'  {label.replace(chr(10), " "):20s}  손실: {final:.6f}  |  인코더 파라미터: {n_params}')

## 9. 매니폴드 연속성 정량 평가

**색상 연속성 지표** — 2D 투영 좌표의 이웃과 원본 매니폴드 파라미터 t의 순위 상관  
→ Spearman 상관이 1에 가까울수록 "롤이 잘 펼쳐진 것"

In [ ]:
# ─── 매니폴드 연속성 평가: t와 2D 좌표의 순위 상관 ───
from scipy.stats import spearmanr
from sklearn.neighbors import NearestNeighbors

def continuity_score(coords_2d, t_vals, k=10):
    """
    2D 투영에서 k-이웃의 매니폴드 파라미터 t 분산 평균
    → 작을수록 이웃 점들이 매니폴드 위에서도 가까움 (잘 펼쳐진 것)
    """
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords_2d)
    _, indices = nbrs.kneighbors(coords_2d)
    # 각 점의 k-이웃 t값 분산 평균
    neighbor_t_var = np.mean([
        np.var(t_vals[indices[i, 1:]])   # 자기 자신 제외
        for i in range(len(coords_2d))
    ])
    return neighbor_t_var


print('=== 매니폴드 연속성 평가 ===')
print('(이웃 t값 분산: 낮을수록 = 매니폴드가 잘 펼쳐진 것)')
print()

all_labels  = ['PCA'] + [l.replace('\n', ' ') for l in results.keys()] + ['Isomap']
all_coords  = [coords_pca] + [d['coords'] for d in results.values()] + [coords_isomap]
score_colors = ['#95a5a6', '#bdc3c7', '#3498DB', '#2ECC71', '#E74C3C', '#9B59B6']

continuity_scores = []
for label, coords in zip(all_labels, all_coords):
    score = continuity_score(coords, t)
    continuity_scores.append(score)
    best = ' ← 최소 (최고)' if score == min([continuity_score(c, t) for c in all_coords]) else ''
    print(f'  {label:25s}  이웃 t분산: {score:.4f}{best}')

# 막대 차트
fig_cont = go.Figure(go.Bar(
    x=all_labels,
    y=continuity_scores,
    marker_color=score_colors,
    opacity=0.85,
    text=[f'{s:.4f}' for s in continuity_scores],
    textposition='auto',
))
fig_cont.update_layout(
    title='매니폴드 연속성 평가 — 2D 이웃의 t값 분산 (↓ 낮을수록 잘 펼쳐진 것)',
    xaxis_title='기법',
    yaxis_title='이웃 t값 평균 분산 (낮을수록 좋음)',
    width=800, height=450,
    showlegend=False,
)
fig_cont.show()

## 10. Bottleneck 공간 탐색: hidden 노드 활성화 시각화

MLP-16에서 hidden layer의 16개 뉴런이 각각 무엇을 감지하는지 확인합니다.

In [ ]:
# ─── MLP-16 hidden 뉴런 활성화 시각화 ───
# 가장 큰 MLP-16 인코더의 hidden layer 출력 추출
enc_mlp16_model = results['MLP-16\n(hidden=16)']['encoder']

# hidden layer 출력을 뽑는 중간 모델 생성
hidden_layer = enc_mlp16_model.get_layer('인코더_hidden_16')
hidden_extractor = keras.Model(
    inputs=enc_mlp16_model.input,
    outputs=hidden_layer.output,
)
hidden_activations = hidden_extractor.predict(X, verbose=0)  # (N, 16)

print(f'hidden 활성화 shape: {hidden_activations.shape}')
print(f'ReLU 이후 0인 비율 (dead neuron): {(hidden_activations == 0).mean():.2%}')

# 활성화된 뉴런 상위 6개를 Swiss Roll 색상으로 표시
active_ratio = (hidden_activations > 0).mean(axis=0)
top6_idx = np.argsort(active_ratio)[::-1][:6]

coords_16 = results['MLP-16\n(hidden=16)']['coords']

fig_act = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'뉴런 {i} (활성={active_ratio[i]:.0%})' for i in top6_idx],
)

for plot_idx, neuron_idx in enumerate(top6_idx):
    row, col = divmod(plot_idx, 3)
    fig_act.add_trace(
        go.Scatter(
            x=coords_16[:, 0],
            y=coords_16[:, 1],
            mode='markers',
            marker=dict(
                size=3,
                color=hidden_activations[:, neuron_idx],
                colorscale='Viridis',
                opacity=0.8,
                showscale=(plot_idx == 0),
                colorbar=dict(title='활성화값') if plot_idx == 0 else None,
            ),
            showlegend=False,
        ),
        row=row + 1, col=col + 1,
    )

fig_act.update_layout(
    title='MLP-16 인코더 — 상위 6개 뉴런의 활성화 패턴 (2D bottleneck 공간 위에 표시)',
    width=1000, height=650,
)
fig_act.show()
print('각 뉴런이 매니폴드의 서로 다른 영역을 담당하는지 확인')

## 11. 결론

### PCA ↔ 선형 오토인코더 동치성 확인

| 조건 | 선형 AE | PCA |
|------|---------|-----|
| 최적화 목표 | MSE 최소화 | 분산 최대화 |
| 해 | 가중치 행렬 **W** | 주성분 벡터 **V** |
| 관계 | **W ∈ span(V)** — 같은 부분공간 | (위와 동일) |

> 선형 AE의 bottleneck 표현은 PCA와 **동일한 부분공간**을 학습합니다 (회전만 다를 수 있음).

### ReLU MLP의 Swiss Roll 언폴딩

| 모델 | 기대 동작 | 근거 |
|------|-----------|------|
| 선형 AE | PCA와 동일 — 롤이 접혀 보임 | 선형 변환 한계 |
| MLP-4 | 미약한 비선형성 | 표현력 부족 |
| MLP-8 | 부분적 언폴딩 | 비선형 구조 일부 학습 |
| MLP-16 | 더 나은 언폴딩 | 표현력 증가 |
| Isomap | 가장 깔끔한 언폴딩 | 측지 거리 직접 사용 |

### 핵심 통찰

> **MLP 오토인코더는 노드 수가 늘수록 비선형 매니폴드를 더 잘 학습하지만,  
> Isomap처럼 "데이터 구조를 명시적으로 아는" 기법에는 여전히 미치지 못합니다.**  
>  
> ReLU는 구간별 선형(piecewise linear) 함수이므로,  
> 레이어/노드가 충분히 많아야 Swiss Roll 같은 곡면 구조를 근사할 수 있습니다.

### 더 나아가기
- **Variational Autoencoder (VAE)**: 잠재 공간에 정규화 추가 → 더 매끄러운 manifold
- **더 깊은 네트워크 (층 추가)**: 노드 수보다 깊이가 비선형 표현에 더 효율적
- **Contractive Autoencoder**: Jacobian 정규화로 매니폴드 구조 보존 강화